In [ ]:
# Import required libraries
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd  # For compatibility with some functions
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("📚 Libraries loaded successfully!")
print(f"Using Polars version: {pl.__version__}")


In [ ]:
# Check if PISA data is available
data_dir = Path("data")
pisa_dir = data_dir / "pisa"

# List available files
pisa_files = list(pisa_dir.glob("*.csv")) + list(pisa_dir.glob("*.parquet"))

if not pisa_files:
    print("⚠️ No PISA data files found.")
    print("Please download PISA 2022 data from: https://www.oecd.org/pisa/data/2022database/")
    print("Convert to CSV or Parquet format and place in the data/pisa/ directory")
    
    # Create sample data for demonstration
    print("\n📊 Creating sample data for demonstration...")
    
    # Sample countries for comparison
    countries = ['ESP', 'DEU', 'FRA', 'ITA', 'PRT', 'FIN', 'NLD', 'OECD_AVG']
    country_names = ['Spain', 'Germany', 'France', 'Italy', 'Portugal', 'Finland', 'Netherlands', 'OECD Average']
    
    # Generate sample data with realistic patterns
    np.random.seed(42)
    n_students_per_country = 1000
    
    sample_data = []
    for i, (cnt, name) in enumerate(zip(countries, country_names)):
        # Simulate different patterns for Spain vs others
        if cnt == 'ESP':
            # Spanish students: slightly lower homework time, similar performance
            homework_time = np.random.normal(180, 60, n_students_per_country)  # 3 hours avg
            math_score = np.random.normal(485, 80, n_students_per_country)
            motivation = np.random.normal(0.1, 1.0, n_students_per_country)  # Slightly lower
        else:
            # Other countries
            homework_time = np.random.normal(200, 70, n_students_per_country)  # 3.3 hours avg
            math_score = np.random.normal(500, 85, n_students_per_country)
            motivation = np.random.normal(0.2, 1.0, n_students_per_country)
        
        # Create student records
        for j in range(n_students_per_country):
            sample_data.append({
                'CNT': cnt,
                'country_name': name,
                'student_id': f"{cnt}_{j:04d}",
                'homework_time_min': max(0, homework_time[j]),  # Minutes per week
                'math_score': math_score[j],
                'motivation_index': motivation[j],
                'socioeconomic_status': np.random.normal(0, 1),
                'gender': np.random.choice(['Male', 'Female']),
                'school_belonging': np.random.normal(0, 1),
                'perseverance': np.random.normal(0, 1)
            })
    
    # Create Polars DataFrame
    pisa_df = pl.DataFrame(sample_data)
    print(f"✅ Created sample dataset with {len(pisa_df)} students from {len(countries)} countries")
    
else:
    print(f"📁 Found PISA data files: {[f.name for f in pisa_files]}")
    # Load the actual data (uncomment when you have real data)
    # pisa_df = pl.read_csv(pisa_files[0])  # or pl.read_parquet()

# Display basic info about the dataset
print(f"\nDataset shape: {pisa_df.shape}")
print(f"Countries in dataset: {sorted(pisa_df['country_name'].unique())}")


In [ ]:
# Quick Analysis: Spain vs Other Countries
print("🔍 QUICK ANALYSIS: SPAIN vs INTERNATIONAL COMPARISON")
print("=" * 60)

# Calculate country-level statistics
country_stats = pisa_df.group_by('country_name').agg([
    pl.col('homework_time_min').mean().alias('avg_homework_hours'),
    pl.col('math_score').mean().alias('avg_math_score'),
    pl.col('motivation_index').mean().alias('avg_motivation'),
    pl.col('school_belonging').mean().alias('avg_belonging'),
    pl.count().alias('n_students')
]).with_columns([
    (pl.col('avg_homework_hours') / 60).alias('avg_homework_hours')  # Convert to hours
]).sort('avg_homework_hours', descending=True)

print("\n📊 Country-Level Summary Statistics:")
print(country_stats)

# Focus on Spain
spain_stats = country_stats.filter(pl.col('country_name') == 'Spain')
if len(spain_stats) > 0:
    print(f"\n🇪🇸 Spain's Profile:")
    print(f"Homework time: {spain_stats['avg_homework_hours'][0]:.1f} hours/week")
    print(f"Math score: {spain_stats['avg_math_score'][0]:.1f}")
    print(f"Motivation index: {spain_stats['avg_motivation'][0]:.2f}")
    print(f"School belonging: {spain_stats['avg_belonging'][0]:.2f}")

# Quick visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Convert to pandas for easier plotting
stats_pd = country_stats.to_pandas()
colors = ['red' if x == 'Spain' else 'lightblue' for x in stats_pd['country_name']]

# Homework time comparison
axes[0].bar(stats_pd['country_name'], stats_pd['avg_homework_hours'], color=colors)
axes[0].set_title('Average Homework Time per Week')
axes[0].set_ylabel('Hours per week')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# Math performance comparison
axes[1].bar(stats_pd['country_name'], stats_pd['avg_math_score'], color=colors)
axes[1].set_title('Average Math Performance')
axes[1].set_ylabel('PISA Score')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Findings Summary:")
print("- Red bars show Spain's position relative to other countries")
print("- This analysis uses objective PISA data to measure engagement patterns")
print("- Cultural context should be considered when interpreting differences")
